# 10 YOLO 标注导入

**用途**：将 YOLO 格式 `.txt` 标注导入 FiftyOne。

支持两种模式：
- `create_new`：从 YOLO 数据集目录新建 dataset
- `add_field`：向已有 dataset 追加标注字段


In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

MODE         = "add_field"   # "create_new" 或 "add_field"
DATASET_NAME = "sahi_null_v2_ms1_0605-0621_40_ok"
LABEL_FIELD  = "ground_truth"          # 追加时的字段名（add_field 模式用）
YOLO_DATA_DIR  = Path("/path/to/yolo/dataset")  # 数据集根目录（含 images/ labels/）
CLASSES_PATH   = YOLO_DATA_DIR / "classes.txt"  # 类别文件


In [ ]:
# ── 1. 初始化日志 ──────────────────────────────────────────
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ 10_yolo_label_import 初始化完成 ============")


## Step 1: 验证输入路径，展示 classes 列表


In [ ]:
import fiftyone as fo
import pandas as pd
from IPython.display import display

logger.info("Step 1 开始：验证路径")

if not YOLO_DATA_DIR.exists():
    logger.error(f"YOLO_DATA_DIR 不存在: {YOLO_DATA_DIR}")
    raise FileNotFoundError(f"目录不存在: {YOLO_DATA_DIR}")

if not CLASSES_PATH.exists():
    logger.error(f"classes 文件不存在: {CLASSES_PATH}")
    raise FileNotFoundError(f"classes.txt 不存在: {CLASSES_PATH}")

classes_text = CLASSES_PATH.read_text(encoding="utf-8").strip()
class_list = [line.strip() for line in classes_text.splitlines() if line.strip()]
classes_df = pd.DataFrame({"class_id": range(len(class_list)), "class_name": class_list})
print(f"Classes 列表（共 {len(class_list)} 类）:")
display(classes_df)

logger.info(f"Step 1 完成：共 {len(class_list)} 个类别")


## Step 2A: 新建 dataset（MODE=create_new）


In [ ]:
if MODE == "create_new":
    logger.info("Step 2A 开始：新建 dataset from YOLO 目录")

    ds = fo.Dataset.from_dir(
        dataset_dir=str(YOLO_DATA_DIR),
        dataset_type=fo.types.YOLOv5Dataset,
        name=DATASET_NAME,
    )
    ds.persistent = True
    print(f"已创建数据集: {ds.name}，样本数: {len(ds)}")
    logger.info(f"Step 2A 完成：{ds.name}，{len(ds)} 个样本")
else:
    print(f"当前 MODE='{MODE}'，跳过 Step 2A")
    logger.info("Step 2A 跳过")


## Step 2B: 追加标注到已有 dataset（MODE=add_field）


In [ ]:
if MODE == "add_field":
    logger.info(f"Step 2B 开始：向 {DATASET_NAME} 追加标注字段 '{LABEL_FIELD}'")

    ds = fo.load_dataset(DATASET_NAME)
    labels_dir = YOLO_DATA_DIR / "labels"

    success_count = 0
    skip_count = 0

    for sample in ds.iter_samples(autosave=True):
        img_path = Path(sample.filepath)
        label_path = labels_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            sample[LABEL_FIELD] = fo.Detections(detections=[])
            skip_count += 1
            continue

        try:
            detections = []
            for line in label_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = cx - bw / 2
                y1 = cy - bh / 2
                label_name = class_list[cls_id] if cls_id < len(class_list) else str(cls_id)
                detections.append(fo.Detection(label=label_name, bounding_box=[x1, y1, bw, bh]))
            sample[LABEL_FIELD] = fo.Detections(detections=detections)
            success_count += 1
        except Exception as e:
            logger.warning(f"解析标注失败，跳过: {label_path.name} ({e})")
            sample[LABEL_FIELD] = fo.Detections(detections=[])
            skip_count += 1

    print(f"导入完成：成功 {success_count}，跳过/无标注 {skip_count}")
    logger.info(f"Step 2B 完成：success={success_count}, skip={skip_count}")
else:
    print(f"当前 MODE='{MODE}'，跳过 Step 2B")
    logger.info("Step 2B 跳过")


## Step 3: 验证导入结果


In [ ]:
logger.info("Step 3 开始：验证导入结果")

schema = ds.get_field_schema()
if LABEL_FIELD in schema:
    all_dets = ds.values(f"{LABEL_FIELD}.detections")
    has_label = sum(1 for d in all_dets if d)
    no_label  = len(ds) - has_label
    print(f"有标注样本: {has_label}，无标注样本: {no_label}")

    sample_df = pd.DataFrame({
        "filepath":  ds.values("filepath")[:10],
        "det_count": [len(d) if d else 0 for d in all_dets[:10]],
    })
    display(sample_df)
    logger.info(f"Step 3 完成：有标注 {has_label}/{len(ds)}")
else:
    print(f"字段 '{LABEL_FIELD}' 不存在于数据集中")
    logger.warning(f"Step 3: 字段 '{LABEL_FIELD}' 不存在")


## 打开 App 查看


In [ ]:
session = fo.launch_app(ds)
